# Likelihood cost with free $z$ and $\sigma_\star$, M1_210210, CPU

- Model and likelihood come from the cells of the production fit notebook.
- CPU times guide the search only; GPU ratios differ.

## Build

- `build` runs the settings, data, observation and model cells, then the likelihood lines of the fit cell.
- `free_z` and `free_sigma` drop or keep the two `PRIORS` entries.
- `baked` sets `Spectrum(baked_runtime=...)`; `pad_2n` turns the shorter zero pad off.

In [1]:
import json, os, tempfile, time
from pathlib import Path

os.environ["JAX_PLATFORMS"] = "cpu"
os.environ["MPLBACKEND"] = "Agg"
ROOT = Path.cwd().parents[1] if Path.cwd().name == "runtime-z-sigma-speed" else Path.cwd()
OUT = ROOT / "results/runtime-z-sigma-speed"
os.chdir(ROOT)
os.environ["CERIDWEN_TARGET_ID"] = "M1_210210"
os.environ["CERIDWEN_RESULT_DIR"] = tempfile.mkdtemp()

import jax, jax.numpy as jnp, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ceridwen.observation._smoothing as smoothing

FIT_CELLS = ["".join(c["source"]) for c in json.load(open(ROOT / "notebooks/ceridwen_integrated_photometry_spectra.ipynb"))["cells"]]
SHORT_PAD = smoothing._padded_length
BATCH = 100  # num_delete of the production sampler: particles per vmapped call
SEED = 20260921


def build(free_z, free_sigma, baked=False, pad_2n=False):
    smoothing._padded_length = (lambda n, sigma_pix: 2 * n) if pad_2n else SHORT_PAD
    cells = list(FIT_CELLS)
    flag = 'free_z="zred" in PRIORS,'
    assert cells[6].count(flag) == 1
    cells[6] = cells[6].replace(flag, f"{flag} baked_runtime={baked},")
    ns = {"display": lambda *a, **k: None}
    exec(cells[2], ns)
    if not free_z:
        ns["PRIORS"].pop("zred")
    if not free_sigma:
        ns["PRIORS"].pop("sigma_smooth")
    for index in (4, 6, 8):
        exec(cells[index], ns)
    exec("calibration_polynomial =" + cells[10].split("calibration_polynomial =")[1].split("model_parameter_block_text")[0], ns)
    plt.close("all")
    return ns


def loglike(ns):  # body of ceridwen.sampler.runner.run_sampler's loglike_fn
    model, likelihood = ns["joint_model"], ns["joint_likelihood"]
    data = {k: (model.obs_dict[k].flux, model.obs_dict[k].uncertainty, model.obs_dict[k].mask) for k in likelihood.keys}
    def fn(theta):
        prediction, total = model.predict(theta), jnp.zeros(())
        for key, term in zip(likelihood.keys, likelihood.likelihoods):
            y, sigma, mask = data[key]
            total = total + term(y, prediction[key], sigma, mask, params=theta)[0]
        return total
    return jax.jit(jax.vmap(fn))


def prior_points(model):
    key = jax.random.PRNGKey(SEED)
    return {
        name: jnp.asarray(model.priors[name].sample(jax.random.fold_in(key, i), (BATCH, *np.shape(template)))).reshape((BATCH, *np.shape(template)))
        for i, (name, template) in enumerate(model.theta_init.items())
    }


def us_per_call(fn, *args, repeats=7):
    jax.block_until_ready(fn(*args))
    times = []
    for _ in range(repeats):
        start = time.perf_counter(); jax.block_until_ready(fn(*args)); times.append(time.perf_counter() - start)
    return np.median(times) / BATCH * 1e6

## Whole likelihood

- Seven arms; 100 prior draws per arm from one key.
- Arms are timed in eight interleaved rounds; the fastest median per arm is kept.
- $\Delta\ln L$ is against the current path with the same free parameters.

In [2]:
ARMS = {  # label: (free_z, free_sigma, baked, pad_2n)
    "fixed z, sigma": (0, 0, False, False),
    "free sigma, current": (0, 1, False, False),
    "free sigma, baked": (0, 1, True, False),
    "free z, current": (1, 0, False, False),
    "free z, baked": (1, 0, True, False),
    "free z, sigma, current": (1, 1, False, False),
    "free z, sigma, baked, 2n pad": (1, 1, True, True),
    "free z, sigma, baked": (1, 1, True, False),
}
REFERENCE = {"free sigma, baked": "free sigma, current", "free z, baked": "free z, current",
             "free z, sigma, baked, 2n pad": "free z, sigma, current", "free z, sigma, baked": "free z, sigma, current"}
compiled, lnl, spectra = {}, {}, {}
for label, arm in ARMS.items():
    ns = build(*arm)
    theta = prior_points(ns["joint_model"])
    compiled[label] = (loglike(ns), theta)
    lnl[label] = np.asarray(compiled[label][0](theta))
    spectra[label] = ns["spectrum_obs"]
best = dict.fromkeys(ARMS, np.inf)
for _ in range(8):
    for label, (fn, theta) in compiled.items():
        best[label] = min(best[label], us_per_call(fn, theta))
timing = pd.DataFrame({"us_per_call": best})
timing["ratio_to_fixed"] = timing["us_per_call"] / best["fixed z, sigma"]
for label, reference in REFERENCE.items():
    timing.loc[label, "max_abs_dlnL"] = np.abs(lnl[label] - lnl[reference]).max()
    timing.loc[label, "max_rel_dlnL"] = np.abs((lnl[label] - lnl[reference]) / lnl[reference]).max()
timing.to_csv(OUT / "timing.csv", index_label="arm")
print("lnL of the draws:", lnl["free z, sigma, current"].min(), "to", lnl["free z, sigma, current"].max())
timing

<string>:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


Using only diffuse dust attenuation.
Initializing DiffuseDust model...
Dust initialization complete.
Spectrum model: dust attenuation only (alpha-enhanced, no nebular)


<string>:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


Using only diffuse dust attenuation.
Initializing DiffuseDust model...
Dust initialization complete.
Spectrum model: dust attenuation only (alpha-enhanced, no nebular)


<string>:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


...

Using only diffuse dust attenuation.
Initializing DiffuseDust model...
Dust initialization complete.
Spectrum model: dust attenuation only (alpha-enhanced, no nebular)


<string>:120: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


Using only diffuse dust attenuation.
Initializing DiffuseDust model...
Dust initialization complete.
Spectrum model: dust attenuation only (alpha-enhanced, no nebular)


lnL of the draws: -931417832.5501584 to 215326.7991603113


,us_per_call,ratio_to_fixed,max_abs_dlnL,max_rel_dlnL
"fixed z, sigma",114.43000,1.000000,NaN,NaN
"free sigma, current",192.05750,1.678384,NaN,NaN
"free sigma, baked",163.38500,1.427816,9.313226e-10,9.185466e-15
"free z, current",198.13417,1.731488,NaN,NaN
"free z, baked",124.97834,1.092182,3.026798e-09,1.843592e-14
"free z, sigma, current",266.83167,2.331833,NaN,NaN
"free z, sigma, baked, 2n pad",205.89417,1.799302,3.259629e-09,3.033686e-15
"free z, sigma, baked",159.89250,1.397295,2.793968e-09,2.329886e-14


## Parts of `Spectrum.predict`

- Same 100 input spectra through each part.
- Smoother grid sizes come from the `sedpy_jax` grid helpers on the trimmed model grid.
- FFT rows time `irfft(rfft(x))` alone at the padded lengths in use.

In [3]:
from sedpy_jax.smoothing import _log_grid, _lsf_grid

fixed, current, baked = (spectra[k] for k in ("fixed z, sigma", "free z, sigma, current", "free z, sigma, baked"))
rng = np.random.default_rng(SEED)
n_model = len(ns["joint_model"].wave)
spec = jnp.asarray(rng.uniform(0.5, 1.5, (BATCH, n_model)), dtype=jnp.float32)
sigma = jnp.full((BATCH,), ns["sigma_star"])
zred = jnp.asarray(rng.uniform(ns["z_catalog"] - 0.1, ns["z_catalog"] + 0.1, BATCH))
mu = jax.vmap(fixed._predict_fn)(spec)
wo = jnp.asarray(current._wavelength)
stretch = lambda interp: jax.jit(jax.vmap(lambda m, z: interp(wo * (1 + current._zred_setup) / (1 + z), m)))
fft = jax.jit(jax.vmap(lambda v: jnp.fft.irfft(jnp.fft.rfft(v), n=v.shape[0])))
parts = {
    "one static Gaussian (fixed sigma)": us_per_call(jax.jit(jax.vmap(fixed._predict_fn)), spec),
    "LOSVD + instrument chain, current": us_per_call(jax.jit(jax.vmap(current._predict_fn)), spec, sigma),
    "LOSVD + instrument chain, baked": us_per_call(jax.jit(jax.vmap(baked._predict_fn)), spec, sigma),
    "redshift stretch, jnp.interp": us_per_call(stretch(current._stretch_interp), mu, zred),
    "redshift stretch, lookup table": us_per_call(stretch(baked._stretch_interp), mu, zred),
}
for method in ("scan_unrolled", "sort"):
    def with_method(x, fp, method=method):
        i = jnp.clip(jnp.searchsorted(wo, x, side="right", method=method), 1, len(wo) - 1)
        return fp[i - 1] + (x - wo[i - 1]) / (wo[i] - wo[i - 1]) * (fp[i] - fp[i - 1])
    parts[f"redshift stretch, searchsorted method={method}"] = us_per_call(stretch(with_method), mu, zred)
for length in (8192, 32768, 18432):
    parts[f"FFT pair, length {length}"] = us_per_call(fft, jnp.asarray(rng.normal(size=(BATCH, length))))
parts = pd.DataFrame({"us_per_call": parts})
parts.to_csv(OUT / "parts.csv", index_label="part")
print("model pixels", n_model, "| observed pixels", len(wo), "| static smoother grid", fixed.smoother_grid_size)
parts

model pixels 10992 | observed pixels 6166 | static smoother grid 4096


,us_per_call
one static Gaussian (fixed sigma),20.22625
"LOSVD + instrument chain, current",104.04708
"LOSVD + instrument chain, baked",62.30209
"redshift stretch, jnp.interp",89.01916
"redshift stretch, lookup table",16.48250
"redshift stretch, searchsorted method=scan_unrolled",234.25625
"redshift stretch, searchsorted method=sort",174.40209
"FFT pair, length 8192",16.68250
"FFT pair, length 32768",66.72625
"FFT pair, length 18432",33.98125
